# Load libraries

In [ ]:
import gc
import ctypes

libc = ctypes.CDLL("libc.so.6")

import cudf
import numpy as np
import pandas as pd

In [ ]:
def free_memory():
    
    _ = gc.collect()
    libc.malloc_trim(0)
#     torch.cuda.empty_cache()

In [ ]:
COMP_LOC = "/kaggle/input/child-mind-institute-detect-sleep-states/"

# Events data

In [ ]:
%%time

PATH = COMP_LOC + "train_events.csv"
events_df = cudf.read_csv(PATH)

print(events_df.shape)
events_df.head(2)

In [ ]:
free_memory()

## Handle nulls

In [ ]:
events.isnull().sum()

In [ ]:
events = events[events['step'].notnull()]
events.shape

In [ ]:
events.isnull().sum()

# Train series

In [ ]:
%%time

PATH = COMP_LOC + "train_series.parquet"
series_df = pd.read_parquet(PATH)

print(series_df.shape)
series_df.head(2)

In [ ]:
free_memory()

In [ ]:
import pytz
default_datetime = "1970-01-01T00:00:00+0000"
series_df['timestamp'] = series_df['timestamp'].fillna(default_datetime)
series_df['timestamp'] = pd.to_datetime(series_df['timestamp'], format='%Y-%m-%dT%H:%M:%S%z')
series_df['timestamp'] = series_df['timestamp'].apply(lambda x: x.astimezone(pytz.utc))
series_df.loc[series_df.timestamp.dt.year==1970,'timestamp'] = pd.NaT

# Test series

In [ ]:
%%time

PATH = COMP_LOC + "test_series.parquet"
series_df = pd.read_parquet(PATH)

print(series_df.shape)
series_df.head(2)

In [ ]:
free_memory()

In [ ]:
series_df.head(10000).to_csv("test_series.csv")

In [ ]:
series_df.head()

# GroupKFold

In [ ]:
from sklearn.model_selection import GroupKFold

N_FOLDS=5
kfold = GroupKFold(N_FOLDS)
group_var = train.Subject
groups=kfold.split(train, groups=group_var)
regs=[]
cvs=[]
for fold, (tr_idx,te_idx ) in enumerate(tqdm(groups, total=N_FOLDS, desc="Folds")):
    tr_idx=pd.Series(tr_idx).sample(n=2000000,random_state=100).values
    reg = ensemble.ExtraTreesRegressor(n_estimators=100, max_depth=7, n_jobs=-1, random_state=3)
    x_tr,y_tr=train.loc[tr_idx,cols],train.loc[tr_idx,pcols]
    x_te,y_te=train.loc[te_idx,cols],train.loc[te_idx,pcols]
    reg.fit(x_tr,y_tr)
    regs.append(reg)
    cv=metrics.average_precision_score(y_te, reg.predict(x_te).clip(0.0,1.0))
    cvs.append(cv)
print(cvs)

# Read events data

In [ ]:
# Events data
events = cudf.read_csv("/kaggle/input/child-mind-institute-detect-sleep-states/train_events.csv")

print(events.shape)
events.head(2)